In [1]:
import pandas as pd

DATA_DIR = '/kaggle/input/datasets/abdullahehdayb/olist-abdullah-practice/'
labeled = pd.read_csv(DATA_DIR + 'labeled_table_notebook2.csv',
                       parse_dates=['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date'])

print(labeled.shape)
labeled['order_purchase_timestamp'].min(), labeled['order_purchase_timestamp'].max()

(96476, 21)


(Timestamp('2016-09-15 12:16:38'), Timestamp('2018-08-29 15:00:37'))

In [2]:
labeled['order_purchase_timestamp'].dt.to_period('M').value_counts().sort_index()

order_purchase_timestamp
2016-09       1
2016-10     270
2016-12       1
2017-01     750
2017-02    1653
2017-03    2546
2017-04    2303
2017-05    3545
2017-06    3135
2017-07    3872
2017-08    4193
2017-09    4150
2017-10    4478
2017-11    7288
2017-12    5513
2018-01    7069
2018-02    6556
2018-03    7003
2018-04    6798
2018-05    6749
2018-06    6096
2018-07    6156
2018-08    6351
Freq: M, Name: count, dtype: int64

In [3]:
train_cutoff = '2018-03-01'
val_cutoff = '2018-06-01'

train = labeled[labeled['order_purchase_timestamp'] < train_cutoff]
val = labeled[(labeled['order_purchase_timestamp'] >= train_cutoff) & (labeled['order_purchase_timestamp'] < val_cutoff)]
test = labeled[labeled['order_purchase_timestamp'] >= val_cutoff]

print("Train:", train.shape, train['order_purchase_timestamp'].min(), "to", train['order_purchase_timestamp'].max())
print("Val:", val.shape, val['order_purchase_timestamp'].min(), "to", val['order_purchase_timestamp'].max())
print("Test:", test.shape, test['order_purchase_timestamp'].min(), "to", test['order_purchase_timestamp'].max())

Train: (57323, 21) 2016-09-15 12:16:38 to 2018-02-28 23:57:55
Val: (20550, 21) 2018-03-01 00:00:00 to 2018-05-31 23:51:24
Test: (18603, 21) 2018-06-01 00:39:55 to 2018-08-29 15:00:37


In [4]:
for name, split in [('Train', train), ('Val', val), ('Test', test)]:
    rate = split['is_late'].mean()
    print(f"{name}: {len(split)} orders, {rate:.4f} late rate ({split['is_late'].sum()} late)")

Train: 57323 orders, 0.0767 late rate (4395 late)
Val: 20550 orders, 0.1174 late rate (2413 late)
Test: 18603 orders, 0.0548 late rate (1019 late)


In [5]:
labeled.groupby(labeled['order_purchase_timestamp'].dt.to_period('M'))['is_late'].mean()

order_purchase_timestamp
2016-09    1.000000
2016-10    0.011111
2016-12    0.000000
2017-01    0.030667
2017-02    0.032063
2017-03    0.055774
2017-04    0.078593
2017-05    0.036107
2017-06    0.038596
2017-07    0.034349
2017-08    0.033150
2017-09    0.052048
2017-10    0.052925
2017-11    0.143112
2017-12    0.083802
2018-01    0.065639
2018-02    0.160006
2018-03    0.213623
2018-04    0.053104
2018-05    0.082383
2018-06    0.013615
2018-07    0.044834
2018-08    0.103921
Freq: M, Name: is_late, dtype: float64

In [6]:
train.to_csv('/kaggle/working/train.csv', index=False)
val.to_csv('/kaggle/working/val.csv', index=False)
test.to_csv('/kaggle/working/test.csv', index=False)
print("Saved train/val/test:", train.shape, val.shape, test.shape)

Saved train/val/test: (57323, 21) (20550, 21) (18603, 21)
